# 01: k-nearest neighbors

**The algorithm.** To classify a point, find the k training rows closest to it and take a vote.

**Where the time goes.** For every query, the distance to every training row: that part is a matrix multiply, which hardware is great at. Then *picking the k smallest* out of hundreds of thousands of distances per query. That part is memory-bound sorting work, and it's where naive implementations lose.

**Data.** Covertype: 581,012 forest plots, 54 features, 7 tree-cover classes. Standardized, float32. 2,000 rows are held out as queries.

In [ ]:
import sys
sys.path.insert(0, ".")  # so bench_utils (next to this notebook) imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ml_in_rust as mr
from bench_utils import (machine_info, timeit, load_covtype, add_speedup,
                         save_results, line_chart, fmt_ms, only_available)

pd.set_option("display.float_format", "{:,.4f}".format)
%matplotlib inline
pd.Series(machine_info())

In [ ]:
import os
QUICK = os.environ.get("QUICK") == "1"  # QUICK=1 ./run_all.sh for a fast smoke run

X, y = load_covtype()
Xq, yq = X[:2000], y[:2000]
X_pool, y_pool = X[2000:], y[2000:]
SIZES = [10_000, 50_000] if QUICK else [10_000, 50_000, 200_000, len(X_pool)]
K = 5

from sklearn.neighbors import KNeighborsClassifier as SkKNN

def engines(k=K):
    return only_available({
        "sklearn": SkKNN(n_neighbors=k, n_jobs=-1),
        "numpy": mr.KNeighborsClassifier(k, backend="numpy"),
        "rust": mr.KNeighborsClassifier(k, backend="rust"),
        "rust-blas": mr.KNeighborsClassifier(k, backend="rust-blas"),
        "mlx": mr.KNeighborsClassifier(k, backend="mlx"),
    })
print(f"{len(X_pool):,} training rows available, {len(Xq):,} queries")

## Race 1: batch prediction as the training set grows

Each engine predicts all 2,000 queries. Times are the median of 3 runs after a warm-up call. "vs sklearn" above 1 means faster than scikit-learn.

In [ ]:
rows = []
for n in SIZES:
    ref_pred = None
    for name, model in engines().items():
        model.fit(X_pool[:n], y_pool[:n])
        t = timeit(lambda: model.predict(Xq))
        pred = model.predict(Xq)
        if name == "sklearn":
            ref_pred = pred
        rows.append({"train rows": n, "engine": name, "predict s": t,
                     "accuracy": np.mean(pred == yq), "agrees with sklearn": np.mean(pred == ref_pred)})
batch = add_speedup(pd.DataFrame(rows), "predict s", ["train rows"])
batch.pivot(index="train rows", columns="engine", values="vs sklearn").round(2)

In [ ]:
line_chart(batch, "train rows", "predict s", "KNN: time to predict 2,000 rows (lower is faster)",
           "training rows", "seconds (log scale)")
plt.show()
batch

## Race 2: one prediction at a time

A robot, a game or a live API predicts one row at a time. Here the fixed cost of each call matters more than raw math speed.

In [ ]:
lat_rows = []
for n in ([10_000] if QUICK else [10_000, 200_000]):
    for name, model in engines().items():
        model.fit(X_pool[:n], y_pool[:n])
        model.predict(Xq[:1])
        times = []
        for i in range(300):
            q = Xq[i:i + 1]
            t0 = __import__("time").perf_counter()
            model.predict(q)
            times.append(__import__("time").perf_counter() - t0)
        lat_rows.append({"train rows": n, "engine": name, "median µs": np.median(times) * 1e6,
                         "p95 µs": np.percentile(times, 95) * 1e6})
latency = add_speedup(pd.DataFrame(lat_rows), "median µs", ["train rows"])
latency

## Race 3: does k change the picture?

The Rust engines keep the k best candidates in a small sorted list, which gets more expensive as k grows. NumPy and MLX use a partial sort whose cost barely depends on k. This race checks whether that matters in practice.

In [ ]:
n = 50_000 if QUICK else 200_000
k_rows = []
for k in [1, 5, 25, 100]:
    for name, model in engines(k).items():
        model.fit(X_pool[:n], y_pool[:n])
        k_rows.append({"k": k, "engine": name, "predict s": timeit(lambda: model.predict(Xq), repeats=2)})
by_k = add_speedup(pd.DataFrame(k_rows), "predict s", ["k"])
line_chart(by_k, "k", "predict s", f"KNN: predict time as k grows ({n:,} training rows)",
           "k (neighbors)", "seconds (log scale)")
plt.show()
by_k.pivot(index="k", columns="engine", values="vs sklearn").round(2)

In [ ]:
save_results("knn", batch, notes={
    "latency": latency.to_dict(orient="records"),
    "by_k": by_k.to_dict(orient="records"),
})
largest = batch[batch["train rows"] == batch["train rows"].max()].set_index("engine")
print("At the largest size, speed vs scikit-learn:")
print(largest["vs sklearn"].round(2).to_string())